In [ ]:
import pandas as pd
%pip install scikit-learn
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(
    '../US_Accidents_March23.csv',
    usecols=['Severity', 'Start_Time', 'Start_Lat', 'Start_Lng',
              'Temperature(F)', 'Humidity(%)', 'Visibility(mi)',
              'Wind_Speed(mph)', 'Precipitation(in)',
              'Weather_Condition', 'Sunrise_Sunset'],
    nrows=200000
)

print("Initial shape:", df.shape)


df['Severity_Binary'] = df['Severity'].apply(lambda x: 0 if x in [1, 2] else 1)
print("\nBinary severity distribution:")
print(df['Severity_Binary'].value_counts())
print(df['Severity_Binary'].value_counts(normalize=True).round(3))


df = df.drop(columns=['Severity'])


df = df.drop(columns=['Precipitation(in)'])


df = df.dropna(subset=['Weather_Condition', 'Sunrise_Sunset'])



for col in ['Wind_Speed(mph)', 'Visibility(mi)', 'Humidity(%)', 'Temperature(F)']:
    df[col] = df[col].fillna(df[col].median())

print("\nMissing values after cleaning:")
print(df.isnull().sum())


df['Start_Time'] = pd.to_datetime(df['Start_Time'])
df['Hour'] = df['Start_Time'].dt.hour
df = df.drop(columns=['Start_Time'])  # no longer needed after extracting Hour


df['Sunrise_Sunset'] = df['Sunrise_Sunset'].map({'Day': 0, 'Night': 1})


top_weather = df['Weather_Condition'].value_counts().nlargest(10).index
df['Weather_Condition'] = df['Weather_Condition'].where(
    df['Weather_Condition'].isin(top_weather), 'Other'
)
df = pd.get_dummies(df, columns=['Weather_Condition'], drop_first=True)


scaler = StandardScaler()
numeric_cols = ['Temperature(F)', 'Humidity(%)', 'Visibility(mi)', 'Wind_Speed(mph)']
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])


# Remove outliers using IQR, but skip columns where it would be degenerate
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    if IQR == 0:
        print(f"Skipping outlier removal for '{col}' — IQR is 0 (data too concentrated)")
        continue
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df = df[(df[col] >= lower) & (df[col] <= upper)]

print("\nFinal shape after outlier removal:", df.shape)

print("\nFinal shape after outlier removal:", df.shape)
print("\nFinal columns:", df.columns.tolist())
print("\nMissing values check (should all be 0):")
print(df.isnull().sum().sum())


df.to_csv('../data/processed/accidents_cleaned.csv', index=False)
print("\nSaved to data/processed/accidents_cleaned.csv")
df.head()


import joblib
joblib.dump(scaler,'../app/models/feature_scaler.pkl')
print("Saved StandardScaler to ../app/models/feature_scaler.pkl")


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Initial shape: (200000, 11)

Binary severity distribution:
Severity_Binary
0    117964
1     82036
Name: count, dtype: int64
Severity_Binary
0    0.59
1    0.41
Name: proportion, dtype: float64

Missing values after cleaning:
Start_Time           0
Start_Lat            0
Start_Lng            0
Temperature(F)       0
Humidity(%)          0
Visibility(mi)       0
Wind_Speed(mph)      0
Weather_Condition    0
Sunrise_Sunset       0
Severity_Binary      0
dtype: int64

Final shape after outlier removal: (155744, 19)

Final columns: ['Start_Lat', 'Start_Lng', 'Temperature(F)', 'Humidity(%)', 'Visibility(mi)', 'Wind_Speed(mph)', 'Sunrise_Sunset', 'Severity_Binary', 'Hour', 'Weather_Condition_Fair', 'Weather_Condition_Haze', 'Weather_Condition_Light Rain', 'Weather_Condition_Light Snow', 'Weather_Condition_Mostly Clo